### Step 1 — Configure project root and Python path

Purpose: Locate the repository root by walking up from the current working directory, then add `src/` to `sys.path` so `ibnr_utils` modules are importable without installation.  
Uses: `pathlib.Path`, `sys.path`.  
Produces: `project_root` — a `Path` object pointing to the repository root.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "src" / "ibnr_utils").exists():
            return path
    raise FileNotFoundError("Could not find IBNR_Utils project root.")


project_root = find_project_root(Path.cwd())
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("Project root:", project_root)

Project root: /mnt/data/Linux/Documents/IBNR_Utils


### Step 2 — Import ibnr_utils functions

Purpose: Import the three functions used in this notebook.  
Uses: `database_to_triangle_collection`, `read_triangle_database_csv`, `validate_all`.  
Produces: Module symbols available in the session.

In [2]:
from ibnr_utils import (
    database_to_triangle_collection,
    read_triangle_database_csv,
    validate_all,
)

### Step 3 — Resolve and verify the input path

Purpose: Build a `Path` object to the source CSV and confirm the file exists before attempting to read it.  
Uses: `pathlib.Path`.  
Produces: `database_path` — a verified `Path` to the long-format monthly database.

In [3]:
database_path = project_root / "data" / "input" / "generated_monthly_triangles_database.csv"

if not database_path.exists():
    raise FileNotFoundError(database_path)

database_path

PosixPath('/mnt/data/Linux/Documents/IBNR_Utils/data/input/generated_monthly_triangles_database.csv')

### Step 4 — Load the monthly triangle database

Purpose: Parse the CSV into a standardised long-format DataFrame with the project data contract columns.  
Uses: `read_triangle_database_csv`.  
Produces: `monthly_database` — a DataFrame with columns `concept`, `basis`, `amount_type`, `level_1`–`level_3`, `accident_period`, `development_period`, `amount`.

In [4]:
monthly_database = read_triangle_database_csv(database_path)
monthly_database.head()

,concept,basis,amount_type,level_1,level_2,level_3,accident_period,development_period,amount
0,Exposure,month,cumulative,Auto,Physical Damage,Private Passenger,2016-01,0,912.0
1,Exposure,month,cumulative,Auto,Physical Damage,Private Passenger,2016-02,0,940.9
2,Exposure,month,cumulative,Auto,Physical Damage,Private Passenger,2016-03,0,980.1
3,Exposure,month,cumulative,Auto,Physical Damage,Private Passenger,2016-04,0,1010.0
4,Exposure,month,cumulative,Auto,Physical Damage,Private Passenger,2016-05,0,1050.6


### Step 5 — Inspect database dimensions and contents

Purpose: Confirm row count, the single `basis` value, and the full set of available concepts.  
Uses: `DataFrame.shape`, `DataFrame.unique()`.  
Produces: A tuple displaying row count, basis array, and sorted concept list.

In [5]:
monthly_database.shape, monthly_database["basis"].unique(), sorted(monthly_database["concept"].unique())

((65340, 9),
 <StringArray>
 ['month']
 Length: 1, dtype: str,
 ['ALAE',
  'Claims Paid',
  'Claims Reported',
  'Earned Premium',
  'Exposure',
  'Loss Incurred',
  'Loss Paid',
  'Salvages',
  'Subrogation'])

### Step 6 — Convert database to triangle collection

Purpose: Pivot the long-format database into one upper-triangular DataFrame per concept, indexed by `accident_period` with columns `dev_0`, `dev_1`, …  
Uses: `database_to_triangle_collection`.  
Produces: `monthly_triangles` — a dict mapping concept name to triangle DataFrame.

In [6]:
monthly_triangles = database_to_triangle_collection(monthly_database)
list(monthly_triangles)

['ALAE',
 'Claims Paid',
 'Claims Reported',
 'Earned Premium',
 'Exposure',
 'Loss Incurred',
 'Loss Paid',
 'Salvages',
 'Subrogation']

### Step 7 — Inspect the Loss Incurred triangle

Purpose: Display the Loss Incurred triangle to verify row/column structure, `NaN` pattern, and that development columns span the expected range.  
Produces: Loss Incurred triangle DataFrame (rows = accident months, columns = development months).

In [7]:
monthly_triangles["Loss Incurred"]

,dev_0,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,...,dev_110,dev_111,dev_112,dev_113,dev_114,dev_115,dev_116,dev_117,dev_118,dev_119
accident_period,,,,,,,,,,,,,,,,,,,,,
2016-01,153700.711642,168265.054672,205478.444379,184336.168001,201248.543480,196453.059025,176881.337833,200214.346199,187578.698914,190075.991440,...,177018.978866,171085.713030,173585.908595,169160.747063,173081.527124,176445.361233,172720.085632,173112.466318,174926.788130,175796.196292
2016-02,174526.085249,179397.620865,205041.059805,195258.548978,187048.160649,201436.435289,199293.923502,205210.386921,242546.688042,200341.751962,...,183520.470365,177750.670575,178954.148728,181846.011293,186153.778295,183854.533150,183737.684382,180857.272493,183746.248505,NaN
2016-03,139977.733603,194087.931093,215736.346315,175783.066907,217515.571324,183537.102418,202607.299157,199878.253026,206739.049289,236638.405603,...,185889.013660,185870.257200,187689.323174,188042.155009,189089.878262,185853.845022,186063.868425,184100.501488,NaN,NaN
2016-04,179441.145220,171328.983083,197378.848951,208702.402696,208493.696364,223233.698165,187690.613593,253992.426802,225375.117705,243903.465956,...,200265.915955,199299.153969,197234.175592,195849.823081,193656.903165,201506.058677,196951.796202,NaN,NaN,NaN
2016-05,185553.143033,216977.705207,209628.471276,214214.723923,232801.660391,210101.993900,201936.310336,223423.445555,230653.011303,211583.631794,...,209927.848694,207676.902404,209507.332802,209279.782933,205822.853384,201165.375549,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,362141.012374,432661.332612,445636.521558,541592.036871,378209.883552,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,367005.573515,415446.207539,433412.965713,518520.051557,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,345630.285842,346963.701868,393892.055949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 8 — Validate triangle relationships

Purpose: Run all built-in cross-concept checks on the triangle collection and confirm no anomalies are flagged.  
Uses: `validate_all`.  
Produces: A single pass confirmation printed on success; raises `ValueError` on the first failing check.  

Interpretation: Validation covers paid-vs-incurred ordering, claims-paid-vs-reported ordering, and non-negative incremental checks. A clean pass confirms the monthly triangles are internally consistent before any downstream calculations.

In [8]:
validate_all(monthly_triangles)

All validations passed.
